In [1]:
from pathlib import Path

import matplotlib.pyplot as pl
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split

import shap

c:\Users\trini\OneDrive\Documents\SC4052\SC4052 Project2\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import kagglehub

path = kagglehub.dataset_download("paololol/league-of-legends-ranked-matches")

print("Path to dataset files:", path)

100%|██████████| 186M/186M [00:47<00:00, 4.13MB/s] 

Extracting files...


Path to dataset files: C:\Users\trini\.cache\kagglehub\datasets\paololol\league-of-legends-ranked-matches\versions\9


In [ ]:
# path: C:\Users\trini\.cache\kagglehub\datasets\paololol\league-of-legends-ranked-matches\versions\9

path = Path(path)
matches = pd.read_csv(path/"matches.csv")
participants = pd.read_csv(path / "participants.csv")
stats1 = pd.read_csv(path / "stats1.csv", low_memory=False)
stats2 = pd.read_csv(path / "stats2.csv", low_memory=False)
stats = pd.concat([stats1, stats2])

In [75]:
a = pd.merge(participants, matches, left_on="matchid", right_on="id", suffixes=("", "_matches"))
allstats_orig = pd.merge(a, stats, left_on="matchid", right_on="id", suffixes=("", "_stats"))

In [76]:
allstats = allstats_orig.copy()

In [77]:
allstats

,id,matchid,player,championid,ss1,ss2,role,position,id_matches,gameid,...,neutralminionskilled,ownjunglekills,enemyjunglekills,totcctimedealt,champlvl,pinksbought,wardsbought,wardsplaced,wardskilled,firstblood
0,9,10,1,19,4,11,NONE,JUNGLE,10,3187427022,...,1,1,0,211,14,1,0,17,3,0
1,10,10,2,267,3,4,DUO_SUPPORT,BOT,10,3187427022,...,1,1,0,211,14,1,0,17,3,0
2,11,10,3,119,7,4,DUO_CARRY,BOT,10,3187427022,...,1,1,0,211,14,1,0,17,3,0
3,12,10,4,114,12,4,SOLO,TOP,10,3187427022,...,1,1,0,211,14,1,0,17,3,0
4,13,10,5,112,4,3,SOLO,MID,10,3187427022,...,1,1,0,211,14,1,0,17,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1579405,1865600,187588,6,222,4,7,DUO_CARRY,BOT,187588,1990458004,...,3,1,2,1473,15,0,0,17,6,0
1579406,1865601,187588,7,1,14,4,SOLO,MID,187588,1990458004,...,3,1,2,1473,15,0,0,17,6,0
1579407,1865602,187588,8,53,4,3,DUO_SUPPORT,BOT,187588,1990458004,...,3,1,2,1473,15,0,0,17,6,0
1579408,1865603,187588,9,92,4,11,NONE,JUNGLE,187588,1990458004,...,3,1,2,1473,15,0,0,17,6,0


In [78]:
# drop games that lasted less than 10 minutes
allstats = allstats.loc[allstats["duration"] >= 10 * 60, :]

# drop columns that are not useful for training
cols_to_drop = ['id', 'matchid', 'player', 'id_matches', 'gameid', 'id_stats', 'creation', 'version', 'platformid']
allstats.drop(columns=cols_to_drop, inplace=True)

allstats.shape

C:\Users\trini\AppData\Local\Temp\ipykernel_37632\4163527027.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  allstats.drop(columns=cols_to_drop, inplace=True)


(1537132, 63)

In [79]:
display(allstats.dtypes)

championid      int64
ss1             int64
ss2             int64
role           object
position       object
                ...  
pinksbought     int64
wardsbought    object
wardsplaced     int64
wardskilled     int64
firstblood      int64
Length: 63, dtype: object

In [80]:
allstats.columns

Index(['championid', 'ss1', 'ss2', 'role', 'position', 'queueid', 'seasonid',
       'duration', 'win', 'item1', 'item2', 'item3', 'item4', 'item5', 'item6',
       'trinket', 'kills', 'deaths', 'assists', 'largestkillingspree',
       'largestmultikill', 'killingsprees', 'longesttimespentliving',
       'doublekills', 'triplekills', 'quadrakills', 'pentakills',
       'legendarykills', 'totdmgdealt', 'magicdmgdealt', 'physicaldmgdealt',
       'truedmgdealt', 'largestcrit', 'totdmgtochamp', 'magicdmgtochamp',
       'physdmgtochamp', 'truedmgtochamp', 'totheal', 'totunitshealed',
       'dmgselfmit', 'dmgtoobj', 'dmgtoturrets', 'visionscore', 'timecc',
       'totdmgtaken', 'magicdmgtaken', 'physdmgtaken', 'truedmgtaken',
       'goldearned', 'goldspent', 'turretkills', 'inhibkills',
       'totminionskilled', 'neutralminionskilled', 'ownjunglekills',
       'enemyjunglekills', 'totcctimedealt', 'champlvl', 'pinksbought',
       'wardsbought', 'wardsplaced', 'wardskilled', 'firstblood

In [81]:
cat_cols = allstats.select_dtypes(include='object').columns.tolist()
cat_cols

['role', 'position', 'wardsbought']

In [82]:
# wardsbought should be converted to numeric
allstats["wardsbought"] = allstats["wardsbought"].astype(np.int32)

C:\Users\trini\AppData\Local\Temp\ipykernel_37632\2802020737.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  allstats["wardsbought"] = allstats["wardsbought"].astype(np.int32)


In [83]:
cat_cols = allstats.select_dtypes(include='object').columns.tolist()
cat_cols

['role', 'position']

In [84]:
# role: SOLO / NONE (for jungle) / DUO_CARRY / DUO_SUPPORT
# position: BOT / JUNGLE / TOP / MID

# change position to TOP / JUNGLE / MID / BOT / SUPPORT
allstats['position'] = allstats.apply(
    lambda row: 'SUPPORT' if row['role'] == 'DUO_SUPPORT' else row['position'], 
    axis=1
)

# verify
print(allstats['position'].value_counts())

position
BOT        321759
JUNGLE     314660
MID        305183
TOP        303952
SUPPORT    291578
Name: count, dtype: int64


C:\Users\trini\AppData\Local\Temp\ipykernel_37632\3569718408.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  allstats['position'] = allstats.apply(


In [85]:
allstats.drop(columns=['role'], inplace=True)

C:\Users\trini\AppData\Local\Temp\ipykernel_37632\147914273.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  allstats.drop(columns=['role'], inplace=True)


In [87]:
# encode the 'position' column before training
position_mapping = {
    'TOP': 0,
    'JUNGLE': 1,
    'MID': 2,
    'BOT': 3,
    'SUPPORT': 4
}

allstats['position'] = allstats['position'].map(position_mapping)

C:\Users\trini\AppData\Local\Temp\ipykernel_37632\2371079068.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  allstats['position'] = allstats['position'].map(position_mapping)


In [89]:
# adding cs_per_min
allstats["cs_per_min"] = (allstats["totminionskilled"] + allstats["neutralminionskilled"]) / (allstats["duration"] / 60)

C:\Users\trini\AppData\Local\Temp\ipykernel_37632\3040401597.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  allstats["cs_per_min"] = (allstats["totminionskilled"] + allstats["neutralminionskilled"]) / (allstats["duration"] / 60)


In [90]:
allstats

,championid,ss1,ss2,position,queueid,seasonid,duration,win,item1,item2,...,ownjunglekills,enemyjunglekills,totcctimedealt,champlvl,pinksbought,wardsbought,wardsplaced,wardskilled,firstblood,cs_per_min
0,19,4,11,1,420,8,1909,0,2301,3111,...,1,0,211,14,1,0,17,3,0,0.565741
1,267,3,4,4,420,8,1909,0,2301,3111,...,1,0,211,14,1,0,17,3,0,0.565741
2,119,7,4,3,420,8,1909,0,2301,3111,...,1,0,211,14,1,0,17,3,0,0.565741
3,114,12,4,0,420,8,1909,0,2301,3111,...,1,0,211,14,1,0,17,3,0,0.565741
4,112,4,3,2,420,8,1909,0,2301,3111,...,1,0,211,14,1,0,17,3,0,0.565741
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1579405,222,4,7,3,4,5,2534,0,3401,2049,...,1,2,1473,15,0,0,17,6,0,2.107340
1579406,1,14,4,2,4,5,2534,0,3401,2049,...,1,2,1473,15,0,0,17,6,0,2.107340
1579407,53,4,3,4,4,5,2534,0,3401,2049,...,1,2,1473,15,0,0,17,6,0,2.107340
1579408,92,4,11,1,4,5,2534,0,3401,2049,...,1,2,1473,15,0,0,17,6,0,2.107340


### Training

In [91]:
X = allstats.drop(columns=["win"])
y = allstats["win"]

In [92]:
# create train/validation split
Xt, Xv, yt, yv = train_test_split(X, y, test_size=0.2, random_state=10)
dt = xgb.DMatrix(Xt, label=yt.values)
dv = xgb.DMatrix(Xv, label=yv.values)

In [93]:
params = {
    "objective": "binary:logistic",
    "base_score": np.mean(yt),
    "eval_metric": "logloss",
}
model = xgb.train(
    params,
    dt,
    num_boost_round=10,
    evals=[(dt, "train"), (dv, "valid")],
    early_stopping_rounds=5,
    verbose_eval=25,
)

[0]	train-logloss:0.56937	valid-logloss:0.56950
[9]	train-logloss:0.33000	valid-logloss:0.33057


In [94]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# get predictions on validation set
y_pred_prob = model.predict(dv)
y_pred = (y_pred_prob >= 0.5).astype(int)

# accuracy
accuracy = accuracy_score(yv, y_pred)
print(f"Accuracy: {accuracy:.2%}")

# more detailed breakdown
print(classification_report(yv, y_pred))

# confusion matrix
print(confusion_matrix(yv, y_pred))

Accuracy: 85.53%
              precision    recall  f1-score   support

           0       0.86      0.85      0.85    153938
           1       0.85      0.86      0.86    153489

    accuracy                           0.86    307427
   macro avg       0.86      0.86      0.86    307427
weighted avg       0.86      0.86      0.86    307427

[[130417  23521]
 [ 20950 132539]]


In [ ]:
import joblib
import os

os.makedirs('../ml/models', exist_ok=True)

joblib.dump(model, '../ml/models/model.pkl')

print("Model saved!")

Model saved!


#### Training by Role

In [96]:
allstats.position.value_counts()

position
3    321759
1    314660
2    305183
0    303952
4    291578
Name: count, dtype: int64

In [97]:
stats_top = allstats[allstats.position == 0]
stats_jungle = allstats[allstats.position == 1] 
stats_mid = allstats[allstats.position == 2]
stats_bot = allstats[allstats.position == 3]
stats_support = allstats[allstats.position == 4]

In [100]:
# Training for TOP 
X_top = stats_top.drop(columns=["win"])
y_top = stats_top["win"]

Xt_top, Xv_top, yt_top, yv_top = train_test_split(X_top, y_top, test_size=0.2, random_state=10)
dt_top = xgb.DMatrix(Xt_top, label=yt_top.values)
dv_top = xgb.DMatrix(Xv_top, label=yv_top.values)

params = {
    "objective": "binary:logistic",
    "base_score": np.mean(yt_top),
    "eval_metric": "logloss",
}
model = xgb.train(
    params,
    dt_top,
    num_boost_round=10,
    evals=[(dt_top, "train"), (dv_top, "valid")],
    early_stopping_rounds=5,
    verbose_eval=25,
)

# get predictions on validation set
y_pred_prob_top = model.predict(dv_top)
y_pred_top = (y_pred_prob_top >= 0.5).astype(int)

# accuracy
accuracy_top = accuracy_score(yv_top, y_pred_top)
print(f"Accuracy: {accuracy_top:.2%}")

[0]	train-logloss:0.56982	valid-logloss:0.57046
[9]	train-logloss:0.33133	valid-logloss:0.33489
Accuracy: 85.51%


In [101]:
os.makedirs('../ml/models', exist_ok=True)

joblib.dump(model, '../ml/models/model_top.pkl')
print("Model saved!")

Model saved!


In [102]:
# Training for JUNGLE
X_jungle = stats_jungle.drop(columns=["win"])
y_jungle = stats_jungle["win"]

Xt_jungle, Xv_jungle, yt_jungle, yv_jungle = train_test_split(X_jungle, y_jungle, test_size=0.2, random_state=10)
dt_jungle = xgb.DMatrix(Xt_jungle, label=yt_jungle.values)
dv_jungle = xgb.DMatrix(Xv_jungle, label=yv_jungle.values)

params = {
    "objective": "binary:logistic",
    "base_score": np.mean(yt_jungle),
    "eval_metric": "logloss",
}
model = xgb.train(
    params,
    dt_jungle,
    num_boost_round=10,
    evals=[(dt_jungle, "train"), (dv_jungle, "valid")],
    early_stopping_rounds=5,
    verbose_eval=25,
)

# get predictions on validation set
y_pred_prob_jungle = model.predict(dv_jungle)
y_pred_jungle = (y_pred_prob_jungle >= 0.5).astype(int)

# accuracy
accuracy_jungle = accuracy_score(yv_jungle, y_pred_jungle)
print(f"Accuracy: {accuracy_jungle:.2%}")

[0]	train-logloss:0.56920	valid-logloss:0.57007
[9]	train-logloss:0.32708	valid-logloss:0.33134
Accuracy: 85.58%


In [103]:
os.makedirs('../ml/models', exist_ok=True)

joblib.dump(model, '../ml/models/model_jungle.pkl')
print("Model saved!")

Model saved!


In [104]:
# Training for MID
X_mid = stats_mid.drop(columns=["win"])
y_mid = stats_mid["win"]

Xt_mid, Xv_mid, yt_mid, yv_mid = train_test_split(X_mid, y_mid, test_size=0.2, random_state=10)
dt_mid = xgb.DMatrix(Xt_mid, label=yt_mid.values)
dv_mid = xgb.DMatrix(Xv_mid, label=yv_mid.values)

params = {
    "objective": "binary:logistic",
    "base_score": np.mean(yt_mid),
    "eval_metric": "logloss",
}
model = xgb.train(
    params,
    dt_mid,
    num_boost_round=10,
    evals=[(dt_mid, "train"), (dv_mid, "valid")],
    early_stopping_rounds=5,
    verbose_eval=25,
)

# get predictions on validation set
y_pred_prob_mid = model.predict(dv_mid)
y_pred_mid = (y_pred_prob_mid >= 0.5).astype(int)

# accuracy
accuracy_mid = accuracy_score(yv_mid, y_pred_mid)
print(f"Accuracy: {accuracy_mid:.2%}")

[0]	train-logloss:0.56958	valid-logloss:0.56978
[9]	train-logloss:0.33145	valid-logloss:0.33450
Accuracy: 85.39%


In [105]:
os.makedirs('../ml/models', exist_ok=True)

joblib.dump(model, '../ml/models/model_mid.pkl')
print("Model saved!")

Model saved!


In [106]:
# Training for BOT
X_bot = stats_bot.drop(columns=["win"])
y_bot = stats_bot["win"]

Xt_bot, Xv_bot, yt_bot, yv_bot = train_test_split(X_bot, y_bot, test_size=0.2, random_state=10)
dt_bot = xgb.DMatrix(Xt_bot, label=yt_bot.values)
dv_bot = xgb.DMatrix(Xv_bot, label=yv_bot.values)

params = {
    "objective": "binary:logistic",
    "base_score": np.mean(yt_bot),
    "eval_metric": "logloss",
}
model = xgb.train(
    params,
    dt_bot,
    num_boost_round=10,
    evals=[(dt_bot, "train"), (dv_bot, "valid")],
    early_stopping_rounds=5,
    verbose_eval=25,
)

# get predictions on validation set
y_pred_prob_bot = model.predict(dv_bot)
y_pred_bot = (y_pred_prob_bot >= 0.5).astype(int)

# accuracy
accuracy_bot = accuracy_score(yv_bot, y_pred_bot)
print(f"Accuracy: {accuracy_bot:.2%}")

[0]	train-logloss:0.56896	valid-logloss:0.57057
[9]	train-logloss:0.32905	valid-logloss:0.33690
Accuracy: 85.26%


In [107]:
os.makedirs('../ml/models', exist_ok=True)

joblib.dump(model, '../ml/models/model_bot.pkl')
print("Model saved!")

Model saved!


In [108]:
# Training for SUPPORT
X_support = stats_support.drop(columns=["win"])
y_support = stats_support["win"]

Xt_support, Xv_support, yt_support, yv_support = train_test_split(X_support, y_support, test_size=0.2, random_state=10)
dt_support = xgb.DMatrix(Xt_support, label=yt_support.values)
dv_support = xgb.DMatrix(Xv_support, label=yv_support.values)

params = {
    "objective": "binary:logistic",
    "base_score": np.mean(yt_support),
    "eval_metric": "logloss",
}
model = xgb.train(
    params,
    dt_support,
    num_boost_round=10,
    evals=[(dt_support, "train"), (dv_support, "valid")],
    early_stopping_rounds=5,
    verbose_eval=25,
)

# get predictions on validation set
y_pred_prob_support = model.predict(dv_support)
y_pred_support = (y_pred_prob_support >= 0.5).astype(int)

# accuracy
accuracy_support = accuracy_score(yv_support, y_pred_support)
print(f"Accuracy: {accuracy_support:.2%}")

[0]	train-logloss:0.56920	valid-logloss:0.57002
[9]	train-logloss:0.32788	valid-logloss:0.33317
Accuracy: 85.68%


In [109]:
os.makedirs('../ml/models', exist_ok=True)

joblib.dump(model, '../ml/models/model_support.pkl')
print("Model saved!")

Model saved!
